In [87]:
import sys
import os
import psycopg2
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

load_dotenv()
url = os.getenv("DATABASE_URL")

In [88]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [89]:
query = """
SELECT 
    ticker,
    CASE 
        WHEN EXTRACT(DOW FROM MAX(timestamp)) = 3 THEN MAX(timestamp)
        ELSE NULL
    END AS expiry_date
FROM orderbooks
GROUP BY ticker
"""
cur.execute(query)
rows = cur.fetchall()

data = {ticker: expiry for ticker, expiry in rows}
tickers = [ticker for ticker, date in data.items()]

In [90]:
from datetime import datetime, timezone

#sorted by expiry dates
data = dict(sorted(data.items(), key=lambda item: item[1] if item[1] is not None else datetime(2099,1,1, tzinfo=timezone.utc)))

In [91]:
expiry_dates = [date.replace(hour=0, minute=0, second=0, microsecond=0) for ticker, date in data.items() if date is not None]
unique_dates = list(dict.fromkeys(expiry_dates))
print(unique_dates)

[datetime.datetime(2026, 3, 25, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 8, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 15, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 22, 0, 0, tzinfo=datetime.timezone.utc)]


In [99]:
for date in unique_dates:
    
        #not expiring this day 
    not_expiring_tickers = [ticker for ticker, expiry_date in data.items() 
                             if (expiry_date != None and expiry_date.replace(hour=0, minute=0, second=0, microsecond=0) > date)  
                             or expiry_date == None]

    query = """
        (
            SELECT * 
            FROM orderbooks
            WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
            AND ticker = ANY(%s)
            ORDER BY timestamp ASC 
            LIMIT 1
        )
        UNION ALL
        (
            SELECT * 
            FROM orderbooks 
            WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
            AND ticker = ANY(%s)
            ORDER BY timestamp DESC 
            LIMIT 1
        );
    """

    cur.execute(query, (date, date, not_expiring_tickers, date, date, not_expiring_tickers))
    rows = cur.fetchall()
    print(date)
    print(rows)


2026-03-25 00:00:00+00:00
[(51112, 'SR340CP6', 'OPTSPOT', datetime.datetime(2026, 3, 25, 4, 59, 14, tzinfo=datetime.timezone.utc), [{'price': 0.02, 'quantity': 110}], [], 110, 0), (51065, 'SR300CD6A', 'OPTSPOT', datetime.datetime(2026, 3, 25, 21, 8, 31, tzinfo=datetime.timezone.utc), [{'price': 0.02, 'quantity': 110}], [], 110, 0)]
2026-04-01 00:00:00+00:00
[(146204, 'SR330CD6', 'OPTSPOT', datetime.datetime(2026, 4, 1, 3, 56, 2, tzinfo=datetime.timezone.utc), [{'price': 0.1, 'quantity': 477}], [], 477, 0), (165156, 'SR320CD6B', 'OPTSPOT', datetime.datetime(2026, 4, 1, 21, 8, 16, tzinfo=datetime.timezone.utc), [], [{'price': 1.77, 'quantity': 1}], 0, 1)]
2026-04-08 00:00:00+00:00
[(224005, 'SR330CD6', 'OPTSPOT', datetime.datetime(2026, 4, 8, 5, 14, 22, tzinfo=datetime.timezone.utc), [{'price': 0.1, 'quantity': 369}], [], 369, 0), (246847, 'SR300CP6D', 'OPTSPOT', datetime.datetime(2026, 4, 8, 21, 7, 16, tzinfo=datetime.timezone.utc), [], [{'price': 0.46, 'quantity': 50}], 0, 50)]
2026-04